In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import pickle

def train_and_save_model(data_path, model_path):

    datos = pd.read_csv(data_path)

    if 'Unnamed: 0' in datos.columns:
        datos = datos.drop(columns=['Unnamed: 0'])

    datos['indice_tiempo'] = pd.to_datetime(datos['indice_tiempo'])

    datos['DemandaIndustrialyComercial'] = (
        datos['Demanda Comercial'] + datos['Gran Demanda Industrial/Comercial']
    )

    datos['DemandaTotal'] = (
        datos['DemandaIndustrialyComercial'] + datos['Demanda Residencial']
    )

    df = datos.copy().sort_values("indice_tiempo")

    df["MES"] = df["indice_tiempo"].dt.month

    df["log_PRECIO"] = np.log(df["MONOMICO TOTAL (LOCAL)"])
    df["log_Gas"] = np.log(df["precio GAS NATURAL"])

    # ==============================
    # DEMANDA RELATIVA POR MES
    # ==============================

    df["demanda_prom_mes"] = (
        df.groupby("MES")["DemandaTotal"]
        .transform(lambda x: x.shift().rolling(4).mean())
    )

    df["demanda_rel"] = df["DemandaTotal"] / df["demanda_prom_mes"]

    df["log_demanda_rel"] = np.log(df["demanda_rel"])

    # ==============================
    # HIDRO RELATIVO POR MES
    # ==============================

    df["hidro_prom_mes"] = (
        df.groupby("MES")["Renovable HIDRO > 50"]
        .transform(lambda x: x.shift().rolling(4).mean())
    )

    df["hidro_rel"] = df["Renovable HIDRO > 50"] / df["hidro_prom_mes"]

    df["log_hidro_rel"] = np.log(df["hidro_rel"])

    # ==============================
    # SHARE RENOVABLE
    # ==============================

    df["renov_share"] = (
        df["Generacion Renovable"] /
        (df["Generacion Renovable"] + df["Generacion Termica"])
    )

    df["log_renov_share"] = np.log(df["renov_share"])

    df = df.dropna()

    # ==============================
    # MODELO
    # ==============================

    formula = """
    log_PRECIO ~
    log_Gas +
    log_demanda_rel +
    log_hidro_rel +
    log_renov_share +
    C(MES)
    """

    model = smf.ols(formula=formula, data=df).fit()

    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    print("Modelo guardado en:", model_path)


if __name__ == "__main__":

    train_and_save_model(
        "../data/variables_relevantes_MEM.csv",
        "trained_model.pkl"
    )

Modelo guardado en: trained_model.pkl
